In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
from T_method import LayeredStructure
from My_plotter import Plotter, Style
from Global_optimizer import my_json_load 

In [ ]:
MU_0 = 4e-7 * np.pi
EPS_0 = 8.8541878188e-12
ETA_0 = np.sqrt(MU_0/EPS_0)
C = 1/np.sqrt(MU_0*EPS_0)

In [ ]:
f_min = 3.3e9
f_max = 4.2e9
f_0 = (f_min + f_max)/2
print(f'центральная частота - {(f_0*1.e-9):.2f} ГГц')
lamb_0 = C/f_0
k_0 = 2*np.pi/lamb_0
print(f'длина волны на центральной частоте - {(lamb_0*1000):.2f} мм')
print(f'волновое число на центральной частоте - {(k_0):.2f} 1/м')

In [ ]:
st = Style()
fig, ax = plt.subplots()
pl = Plotter(ax, st)
f = np.array([3.3, 3.5, 3.7, 3.9, 4.2])*1.e9
df = (f - f_0)/f_0
phi = np.array([np.pi/4])
theor_theta = np.linspace(-np.pi/2, np.pi/2, 200)
path = Path(r"results\final\cc10.json")
data = my_json_load(path)
alpha = [3, 0.5, -2]
beta = [2.7, 1.5, 2.7]
alpha_l = 10
alpha_c = 0.01
dipole_shift = np.pi/2
beta_d = np.pi/2

structure = LayeredStructure(alpha, beta=beta, alpha_l=alpha_l, alpha_c=alpha_c, dipole_shift=dipole_shift, beta_d=beta_d)
directivity = 10*np.log10(structure.directivity_two_sources_diagonal(df))
print("directivity=",  directivity)
pl.set_p(f*1.e-9)
for i, f_i in enumerate(f):
    radiation_pattern = structure.radiation_pattern_two_sources_diagonal(phi, theor_theta, np.array([df[i]]), mode='absolute')[0, 0, :]
    pl.multiple_plot(theor_theta*180/np.pi, radiation_pattern, f_i*1.e-9, label=f"f={f_i*1.e-9:.2f} GHz")
pl.set_ylim((-5,100))
pl.set_xlabel("Theta (deg)")
pl.set_ylabel("Radiation Pattern (linear)")
#pl.set_title("Horizontal-plane RP for Different Frequencies")
pl.finalize()
plt.show()

In [ ]:
from MyRadiationPattern import RadiationPattern3DPlotter
f = np.array([3.3, 3.5, 3.7, 3.9, 4.2])*1.e9
df = (f - f_0)/f_0
phi_arr = np.linspace(0, 2*np.pi, 200)
theta_arr = np.linspace(0, np.pi/2, 200)
for i, f_i in enumerate(df):
    dir = structure.radiation_pattern_two_sources_diagonal(phi_arr, theta_arr, np.array([f_i]), mode='absolute')[0, :, :]
    rppl = RadiationPattern3DPlotter(width=500, heigh=700)
    rppl.set_title(f'Frequency={(f[i]*1.e-9):.2f} GHz')
    rppl.plot(phi_arr*180/np.pi, theta_arr*180/np.pi, dir, mode='logarithmic', threshold=-10)

In [ ]:
st = Style()
fig, ax = plt.subplots()
pl1 = Plotter(ax, st)
df = np.linspace(-30, 30, 200)/100

directivity = 10*np.log10(structure.directivity_two_sources_diagonal(df))
pl1.plot((1+df)*f_0*1.e-9, directivity, label=f"Directivity")

pl1.finalize()
pl1.set_ylim((0, 25))
ax.axhline(18, color='gray', linestyle='--', alpha=0.5)
ax.axhline(16, color='gray', linestyle='--', alpha=0.5)
ax.axvline(f_max*1.e-9, color='gray', linestyle='--', alpha=0.5)
ax.axvline(f_min*1.e-9, color='gray', linestyle='--', alpha=0.5)
# ax.axvline((1-0.15)*f_0*1.e-9, color='orange', linestyle='--', alpha=0.5)
# ax.axvline((1+0.15)*f_0*1.e-9, color='orange', linestyle='--', alpha=0.5)
ax.axvline((1)*f_0*1.e-9, color='orange', linestyle='--', alpha=0.5)
ax.set_xlabel('Frequency (GHz)')
ax.set_ylabel('Directivity')
ax.minorticks_on()
plt.show()

In [ ]:
print(f'best_alpha_Z = {ETA_0/alpha} Ohm')
print(f'best_beta_l = {beta/k_0*1000} mm')
print(f'best_alpha_l_Z = {ETA_0/alpha_l} Ohm')
print(f'best_alpha_c_Z = {ETA_0/alpha_c} Ohm')
print(f'best_dipole_shift_l = {dipole_shift/k_0*1000} mm')
print(f'best_beta_d_l = {beta_d/k_0*1000} mm')

In [ ]:
#заготовка для оптимизации 3 листов
seeds = [90, 24]
bounds_items = [["c", [(-20.0, 0)]], ["l", [(0, 20.0)]]]
bounds_beta = [(0.2, 7.0), (0.3, 3.0), (0.3, 3.0)]
l=80
max_summ_beta = np.pi*l/40
for seed in seeds:
    for i1 in bounds_items:
        for i2 in bounds_items:
            for i3 in bounds_items:
                mode_str = i1[0] + i2[0] + i3[0]
                bounds = i1[1] + i2[1] + i3[1]
                result_path = f"results/3_sheets_l_{l}/{mode_str}_seed_{seed}.json"

In [ ]:
#заготовка для оптимизации 2 листов
seeds = [90, 24]
bounds_items = [["c", [(-20.0, 0)]], ["l", [(0, 20.0)]]]
bounds_beta = [(0.2, 7.0), (0.3, 3.0)]
l=80
max_summ_beta = np.pi*l/40
for seed in seeds:
    for i1 in bounds_items:
        for i2 in bounds_items:
            mode_str = i1[0] + i2[0]
            bounds = i1[1] + i2[1]
            result_path = f"results/2_sheets_l_{l}/{mode_str}_seed_{seed}.json"

In [ ]:
print(0.3/k_0*1000)